In [1]:
# JUST ADD IN THE PARAMETER COUNT
# THEN ALSO THE VARIABLE COUNT FROM EARLIER WITH THE SUBTRACTION
# AND APPLY IT TO THE TABLE I HAVE WITH DOMAIN ETC

In [2]:
from process_cellml import process_cellml_file
import pandas as pd
import numpy as np

import hashlib
from pathlib import Path

In [3]:
process_cellml_file('models_cellml2/cell_migration/palecek_horwitz_lauffenburger_1999.cellml')

{'file': 'models_cellml2/cell_migration/palecek_horwitz_lauffenburger_1999.cellml',
 'file_name': 'palecek_horwitz_lauffenburger_1999.cellml',
 'total_equations': 10,
 'diff_eq_raw': 6,
 'alg_eq_raw': 4,
 'ode_like_raw': 0,
 'pde_like_raw': 6,
 'state_count': 0,
 'algebraic_count': 0,
 'warnings': 'Contains <partialdiff/> (PDE-like); libCellML issues present',
 'has_issues': True,
 'raw_variable_count': 114,
 'unique_variable_count': 28,
 'equivalence_edge_count': 172,
 'equivalence_groups': {('environment', 'time'): [('environment', 'time'),
   ('q', 'time'),
   ('s', 'time'),
   ('u', 'time'),
   ('s_', 'time'),
   ('r_', 'time'),
   ('c_', 'time')],
  ('q', 'q'): [('q', 'q'),
   ('u', 'q'),
   ('r_', 'q'),
   ('c_', 'q'),
   ('RT', 'q'),
   ('CT', 'q')],
  ('q', 'kf1'): [('q', 'kf1')],
  ('q', 'kr1'): [('q', 'kr1')],
  ('q', 'kc'): [('q', 'kc'),
   ('s', 'kc'),
   ('u', 'kc'),
   ('s_', 'kc'),
   ('r_', 'kc'),
   ('c_', 'kc'),
   ('parameters', 'kc')],
  ('q', 'kf3'): [('q', 'kf3'),

In [4]:
model_list = pd.read_csv("cellml_links2.csv",index_col=0)
model_list.head()

,0,1,2
0,albrecht_colegrove_friel_2002,https://models.physiomeproject.org/exposure/6c...,calcium_dynamics
1,albrecht_colegrove_hongpaisan_pivovarova_andre...,https://models.physiomeproject.org/exposure/eb...,calcium_dynamics
2,baylor_hollingworth_chandler_2002_a,https://models.physiomeproject.org/exposure/21...,calcium_dynamics
3,baylor_hollingworth_chandler_2002_b,https://models.physiomeproject.org/exposure/21...,calcium_dynamics
4,baylor_hollingworth_chandler_2002_c,https://models.physiomeproject.org/exposure/21...,calcium_dynamics


In [5]:
file_paths = [f"{model_list.iloc[i][2]}/{model_list.iloc[i][0]}.cellml" for i in range(len(model_list))]
domains = model_list["2"]

/var/folders/xr/my72_1yx6g53vqntnttc4v640000gn/T/ipykernel_54373/2162925794.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  file_paths = [f"{model_list.iloc[i][2]}/{model_list.iloc[i][0]}.cellml" for i in range(len(model_list))]


In [6]:
file_paths

['calcium_dynamics/albrecht_colegrove_friel_2002.cellml',
 'calcium_dynamics/albrecht_colegrove_hongpaisan_pivovarova_andrews_friel_2001.cellml',
 'calcium_dynamics/baylor_hollingworth_chandler_2002_a.cellml',
 'calcium_dynamics/baylor_hollingworth_chandler_2002_b.cellml',
 'calcium_dynamics/baylor_hollingworth_chandler_2002_c.cellml',
 'calcium_dynamics/baylor_hollingworth_chandler_2002_d.cellml',
 'calcium_dynamics/baylor_hollingworth_chandler_2002_e.cellml',
 'calcium_dynamics/baylor_hollingworth_chandler_2002_f.cellml',
 'calcium_dynamics/bertram_previte_sherman_kinard_satin_2000_fast.cellml',
 'calcium_dynamics/bertram_previte_sherman_kinard_satin_2000_medium.cellml',
 'calcium_dynamics/bertram_previte_sherman_kinard_satin_2000_slow.cellml',
 'calcium_dynamics/bertram_satin_zhang_smolen_sherman_2004_a.cellml',
 'calcium_dynamics/bertram_satin_zhang_smolen_sherman_2004_b.cellml',
 'calcium_dynamics/bertram_sherman_2004.cellml',
 'calcium_dynamics/bindschadler_sneyd_2001.cellml',
 '

In [7]:
# def batch_process_cellml(files, prefix = "", to_dataframe: bool = True):
#     results = []
#     failed = []

#     for i, fpath in enumerate(files):
#         print(f"[{i+1}/{len(files)}] Processing: {fpath}")
#         try:
#             res = process_cellml_file(f"{prefix}{"/" if prefix is not None else ""}{fpath}")
#             results.append(res)
#         except ValueError as v: 
#             print(v)
#             failed.append(fpath)

#     if to_dataframe:
#         df = pd.DataFrame(results)
#         return df, failed

#     return results, failed

In [8]:
def hash_file(filepath):
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()


def batch_process_cellml(files, prefix="", to_dataframe=True):
    results = []
    failed = []
    seen_hashes = {}

    for i, fpath in enumerate(files):
        full_path = f"{prefix}/{fpath}" if prefix else fpath
        print(f"[{i+1}/{len(files)}] Processing: {fpath}")

        try:
            file_hash = hash_file(full_path)
        except Exception as e:
            print(f"Hashing failed for {fpath}: {e}")
            failed.append(fpath)
            continue

        is_duplicate = file_hash in seen_hashes
        first_seen_path = seen_hashes.get(file_hash)

        if not is_duplicate:
            seen_hashes[file_hash] = fpath

        try:
            res = process_cellml_file(full_path)

            if not isinstance(res, dict):
                res = {"result": res}

            res["file_hash"] = file_hash
            res["file_path"] = fpath
            res["full_path"] = full_path
            res["is_duplicate"] = is_duplicate
            res["duplicate_of"] = first_seen_path if is_duplicate else None

            # optional: infer domain from parent folder
            res["domain"] = Path(fpath).parent.name if Path(fpath).parent.name else None

            results.append(res)

        except ValueError as v:
            print(v)
            failed.append(fpath)

    if to_dataframe:
        df = pd.DataFrame(results)
        return df, failed

    return results, failed

In [9]:
df, failed = batch_process_cellml(file_paths, "models_cellml2")

[1/675] Processing: calcium_dynamics/albrecht_colegrove_friel_2002.cellml
[2/675] Processing: calcium_dynamics/albrecht_colegrove_hongpaisan_pivovarova_andrews_friel_2001.cellml
[3/675] Processing: calcium_dynamics/baylor_hollingworth_chandler_2002_a.cellml
[4/675] Processing: calcium_dynamics/baylor_hollingworth_chandler_2002_b.cellml
[5/675] Processing: calcium_dynamics/baylor_hollingworth_chandler_2002_c.cellml
[6/675] Processing: calcium_dynamics/baylor_hollingworth_chandler_2002_d.cellml
[7/675] Processing: calcium_dynamics/baylor_hollingworth_chandler_2002_e.cellml
[8/675] Processing: calcium_dynamics/baylor_hollingworth_chandler_2002_f.cellml
[9/675] Processing: calcium_dynamics/bertram_previte_sherman_kinard_satin_2000_fast.cellml
[10/675] Processing: calcium_dynamics/bertram_previte_sherman_kinard_satin_2000_medium.cellml
[11/675] Processing: calcium_dynamics/bertram_previte_sherman_kinard_satin_2000_slow.cellml
[12/675] Processing: calcium_dynamics/bertram_satin_zhang_smolen_

In [10]:
failed

[]

In [11]:
def merge_domains_by_hash(df, verbose=False, show_only_duplicates=False, max_print=20):
    """
    Merge rows by file_hash and aggregate domains.

    Parameters:
        verbose (bool): print detailed info
        show_only_duplicates (bool): only print hashes with >1 occurrence
        max_print (int): limit number of hashes printed

    Returns:
        merged DataFrame (one row per unique hash)
    """

    # --- aggregate domains ---
    domain_map = (
        df.groupby("file_hash")["domain"]
        .apply(lambda x: sorted(set(d for d in x if pd.notna(d))))
        .rename("all_domains")
    )

    counts = df["file_hash"].value_counts().rename("n_files_with_same_hash")

    # --- keep one representative row per hash ---
    first_rows = (
        df.sort_values(["file_hash", "is_duplicate"])
        .drop_duplicates(subset="file_hash", keep="first")
        .copy()
    )

    merged = first_rows.merge(domain_map, on="file_hash", how="left")
    merged = merged.merge(counts, on="file_hash", how="left")

    # --- optional printing ---
    if verbose:
        print("\n===== HASH MERGE SUMMARY =====")
        print(f"Total files: {len(df)}")
        print(f"Unique models (by hash): {len(merged)}")
        print(f"Duplicates found: {(counts > 1).sum()}\n")

        # choose which hashes to display
        display_df = merged.copy()
        if show_only_duplicates:
            display_df = display_df[display_df["n_files_with_same_hash"] > 1]

        display_df = display_df.head(max_print)

        print("===== SAMPLE GROUPS =====")
        for _, row in display_df.iterrows():
            print(f"\nHash: {row['file_hash'][:12]}...")
            print(f"Count: {row['n_files_with_same_hash']}")
            print(f"Primary file: {row['file_path']}")
            print(f"Domains: {row['all_domains']}")

            # show all matching files
            matches = df[df["file_hash"] == row["file_hash"]]
            print("All files:")
            for p in matches["file_path"]:
                print(f"  - {p}")

    return merged

In [12]:
merged_df = merge_domains_by_hash(df, verbose=True, show_only_duplicates=True)


===== HASH MERGE SUMMARY =====
Total files: 675
Unique models (by hash): 517
Duplicates found: 133

===== SAMPLE GROUPS =====

Hash: 00f0b69613c5...
Count: 2
Primary file: calcium_dynamics/dawson_lea_irvine_2003.cellml
Domains: ['calcium_dynamics', 'signal_transduction']
All files:
  - calcium_dynamics/dawson_lea_irvine_2003.cellml
  - signal_transduction/dawson_lea_irvine_2003_2.cellml

Hash: 08e62abf6a2f...
Count: 2
Primary file: calcium_dynamics/fridlyand_tamarina_philipson_2003.cellml
Domains: ['calcium_dynamics', 'electrophysiology']
All files:
  - calcium_dynamics/fridlyand_tamarina_philipson_2003.cellml
  - electrophysiology/fridlyand_tamarina_philipson_2003_2.cellml

Hash: 09464143c5e4...
Count: 2
Primary file: circadian_rhythms/ueda_hagiwara_kitano_2001.cellml
Domains: ['circadian_rhythms', 'signal_transduction']
All files:
  - circadian_rhythms/ueda_hagiwara_kitano_2001.cellml
  - signal_transduction/ueda_hagiwara_kitano_2001_2.cellml

Hash: 09b2b6f110d2...
Count: 2
Primary 

In [13]:
merged_df

,file,file_name,total_equations,diff_eq_raw,alg_eq_raw,ode_like_raw,pde_like_raw,state_count,algebraic_count,warnings,...,equivalence_edge_count,equivalence_groups,file_hash,file_path,full_path,is_duplicate,duplicate_of,domain,all_domains,n_files_with_same_hash
0,models_cellml2/calcium_dynamics/dawson_lea_irv...,dawson_lea_irvine_2003.cellml,20,10,10,10,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,114,"{('environment', 'time'): [('environment', 'ti...",00f0b69613c5024d55c13a07fd0016e69efa7b01a25b97...,calcium_dynamics/dawson_lea_irvine_2003.cellml,models_cellml2/calcium_dynamics/dawson_lea_irv...,False,None,calcium_dynamics,"[calcium_dynamics, signal_transduction]",2
1,models_cellml2/circadian_rhythms/goodwin_1965_...,goodwin_1965_a.cellml,2,2,0,2,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,8,"{('environment', 'time'): [('environment', 'ti...",00f497aaea76b568e3615817f3f0fb35fd5e4000aa4284...,circadian_rhythms/goodwin_1965_a.cellml,models_cellml2/circadian_rhythms/goodwin_1965_...,False,None,circadian_rhythms,[circadian_rhythms],1
2,models_cellml2/endocrine/maldonado_2006.cellml,maldonado_2006.cellml,20,9,11,9,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,84,"{('environment', 'time'): [('environment', 'ti...",03b644bfd5cbd12d5a3910bf8be71b920069b09183c49b...,endocrine/maldonado_2006.cellml,models_cellml2/endocrine/maldonado_2006.cellml,False,None,endocrine,[endocrine],1
3,models_cellml2/signal_transduction/riccobene_o...,riccobene_omann_linderman_1999.cellml,8,8,0,8,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,120,"{('environment', 'time'): [('environment', 'ti...",03e82c8edb463012e5adb801864a1ae837e0885ea49806...,signal_transduction/riccobene_omann_linderman_...,models_cellml2/signal_transduction/riccobene_o...,False,None,signal_transduction,[signal_transduction],1
4,models_cellml2/metabolism/olsen_hauser_kummer_...,olsen_hauser_kummer_2003.cellml,29,14,15,14,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,164,"{('environment', 'time'): [('environment', 'ti...",042bbd67fe92f06a738b898e3257db53c436e4e59fe9c5...,metabolism/olsen_hauser_kummer_2003.cellml,models_cellml2/metabolism/olsen_hauser_kummer_...,False,None,metabolism,[metabolism],1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,models_cellml2/electrophysiology/ebihara_johns...,ebihara_johnson_1980.cellml,23,7,16,7,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,56,"{('environment', 'time'): [('environment', 'ti...",f9f15dab19a2987627df45a9b6a987e364a3612f740f09...,electrophysiology/ebihara_johnson_1980.cellml,models_cellml2/electrophysiology/ebihara_johns...,False,None,electrophysiology,[electrophysiology],1
513,models_cellml2/mechanical_constitutive_laws/ri...,rivlin_saunders_1951.cellml,6,0,6,0,0,0,0,libCellML issues present,...,28,"{('interface', 'E11'): [('interface', 'E11'), ...",fd041499ccc21fa9a3fc59964d21c28dcdb06f71b8341c...,mechanical_constitutive_laws/rivlin_saunders_1...,models_cellml2/mechanical_constitutive_laws/ri...,False,None,mechanical_constitutive_laws,[mechanical_constitutive_laws],1
514,models_cellml2/signal_transduction/bagci_2008b...,bagci_2008b.cellml,112,42,70,42,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,464,"{('environment', 'time'): [('environment', 'ti...",fdf927ecdc71473692ddd5e2dba6b0c9ff98bb0c8a57cc...,signal_transduction/bagci_2008b.cellml,models_cellml2/signal_transduction/bagci_2008b...,False,None,signal_transduction,[signal_transduction],1
515,models_cellml2/electrophysiology/maltsev_2009_...,maltsev_2009_paper.cellml,119,29,90,29,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,292,"{('environment', 'time'): [('environment', 'ti...",ff33d53fb395b89046506b82ed2d84fa8d658a08da2b55...,electrophysiology/maltsev_2009_paper.cellml,models_cellml2/electrophysiology/maltsev_2009_...,False,None,electrophysiology,[electrophysiology],1


In [14]:
df = merged_df

In [15]:
#df["domain"] = domains

In [16]:
df.head(2)

,file,file_name,total_equations,diff_eq_raw,alg_eq_raw,ode_like_raw,pde_like_raw,state_count,algebraic_count,warnings,...,equivalence_edge_count,equivalence_groups,file_hash,file_path,full_path,is_duplicate,duplicate_of,domain,all_domains,n_files_with_same_hash
0,models_cellml2/calcium_dynamics/dawson_lea_irv...,dawson_lea_irvine_2003.cellml,20,10,10,10,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,114,"{('environment', 'time'): [('environment', 'ti...",00f0b69613c5024d55c13a07fd0016e69efa7b01a25b97...,calcium_dynamics/dawson_lea_irvine_2003.cellml,models_cellml2/calcium_dynamics/dawson_lea_irv...,False,None,calcium_dynamics,"[calcium_dynamics, signal_transduction]",2
1,models_cellml2/circadian_rhythms/goodwin_1965_...,goodwin_1965_a.cellml,2,2,0,2,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,8,"{('environment', 'time'): [('environment', 'ti...",00f497aaea76b568e3615817f3f0fb35fd5e4000aa4284...,circadian_rhythms/goodwin_1965_a.cellml,models_cellml2/circadian_rhythms/goodwin_1965_...,False,None,circadian_rhythms,[circadian_rhythms],1


In [17]:
df.columns

Index(['file', 'file_name', 'total_equations', 'diff_eq_raw', 'alg_eq_raw',
       'ode_like_raw', 'pde_like_raw', 'state_count', 'algebraic_count',
       'warnings', 'has_issues', 'raw_variable_count', 'unique_variable_count',
       'equivalence_edge_count', 'equivalence_groups', 'file_hash',
       'file_path', 'full_path', 'is_duplicate', 'duplicate_of', 'domain',
       'all_domains', 'n_files_with_same_hash'],
      dtype='object')

In [18]:
df["is_empty_model"] = (
    (df["total_equations"] == 0) &
    (df["ode_like_raw"] == 0) &
    (df["pde_like_raw"] == 0) &
    (df["algebraic_count"] == 0)
)

In [19]:
df[df["is_empty_model"]]


,file,file_name,total_equations,diff_eq_raw,alg_eq_raw,ode_like_raw,pde_like_raw,state_count,algebraic_count,warnings,...,equivalence_groups,file_hash,file_path,full_path,is_duplicate,duplicate_of,domain,all_domains,n_files_with_same_hash,is_empty_model
11,models_cellml2/cardiovascular_circulation/kidn...,kidney_parent.cellml,0,0,0,0,0,0,0,libCellML issues present,...,"{('environment', 'time'): [('environment', 'ti...",090e3fd56c5d49277c91f6c12196b4990abd6709b1c952...,cardiovascular_circulation/kidney_parent.cellml,models_cellml2/cardiovascular_circulation/kidn...,False,None,cardiovascular_circulation,[cardiovascular_circulation],1,True
25,models_cellml2/gene_regulation/PMR2metadata.ce...,PMR2metadata.cellml,0,0,0,0,0,0,0,libCellML issues present,...,{},0c582282cd32256150e688be20f3bc3c2551ab5121387b...,gene_regulation/PMR2metadata.cellml,models_cellml2/gene_regulation/PMR2metadata.ce...,False,None,gene_regulation,"[gene_regulation, synthetic_biology]",2,True
53,models_cellml2/gene_regulation/PMR2_metadata.c...,PMR2_metadata.cellml,0,0,0,0,0,0,0,libCellML issues present,...,{},1b84b48d12c819f7034ffcd3c9136de7df892931895354...,gene_regulation/PMR2_metadata.cellml,models_cellml2/gene_regulation/PMR2_metadata.c...,False,None,gene_regulation,"[gene_regulation, synthetic_biology]",2,True
76,models_cellml2/cardiovascular_circulation/Main...,MainWestkessel.cellml,0,0,0,0,0,0,0,libCellML issues present,...,"{('LEFTHeart', 't'): [('LEFTHeart', 't'), ('Ve...",257f2b07321d53b07b1943856670e7f3fae4e506c18efc...,cardiovascular_circulation/MainWestkessel.cellml,models_cellml2/cardiovascular_circulation/Main...,False,None,cardiovascular_circulation,[cardiovascular_circulation],1,True
102,models_cellml2/gene_regulation/PMR2metadata_2....,PMR2metadata_2.cellml,0,0,0,0,0,0,0,libCellML issues present,...,{},34f678489c8ccd91049970690b52f0ae614d44e3b91348...,gene_regulation/PMR2metadata_2.cellml,models_cellml2/gene_regulation/PMR2metadata_2....,False,None,gene_regulation,"[gene_regulation, synthetic_biology]",2,True
126,models_cellml2/cardiovascular_circulation/pulm...,pulmonary_O2_parent.cellml,0,0,0,0,0,0,0,libCellML issues present,...,"{('environment', 'time'): [('environment', 'ti...",3e6efde01df13956419206865a1b04cccedba3de74cf6e...,cardiovascular_circulation/pulmonary_O2_parent...,models_cellml2/cardiovascular_circulation/pulm...,False,None,cardiovascular_circulation,[cardiovascular_circulation],1,True
160,models_cellml2/cardiovascular_circulation/anti...,antidiuretic_hormone_parent.cellml,0,0,0,0,0,0,0,libCellML issues present,...,"{('environment', 'time'): [('environment', 'ti...",5201eea2bb7cab0a2b1fc5476e2344cc7b4f2a675890fc...,cardiovascular_circulation/antidiuretic_hormon...,models_cellml2/cardiovascular_circulation/anti...,False,None,cardiovascular_circulation,[cardiovascular_circulation],1,True
169,models_cellml2/cardiovascular_circulation/atr_...,atr_natriuretic_peptide_parent.cellml,0,0,0,0,0,0,0,libCellML issues present,...,"{('environment', 'time'): [('environment', 'ti...",559812cebc928f5254dd7a23f3f6096b804a954b02c1f1...,cardiovascular_circulation/atr_natriuretic_pep...,models_cellml2/cardiovascular_circulation/atr_...,False,None,cardiovascular_circulation,[cardiovascular_circulation],1,True
209,models_cellml2/cardiovascular_circulation/M_O2...,M_O2_delivery_parent.cellml,0,0,0,0,0,0,0,libCellML issues present,...,"{('environment', 'time'): [('environment', 'ti...",6774294fe239b496a5f5726da216e69b269a8c7740b223...,cardiovascular_circulation/M_O2_delivery_paren...,models_cellml2/cardiovascular_circulation/M_O2...,False,None,cardiovascular_circulation,[cardiovascular_circulation],1,True
242,models_cellml2/cardiovascular_circulation/pulm...,pulmonary_fluid_parent.cellml,0,0,0,0,0,0,0,libCellML issues present,...,"{('environment', 'time'): [('environment', 'ti...",77edd3b3a03fcd513981365b4799a3439e8c57741ca512...,cardiovascular_circulation/pulmonary_fluid_par...,models_cellml2/cardiovascular_circulation/pulm...,False,None,cardiovascu

In [20]:
df = df[~df["is_empty_model"]]
df.reset_index(drop=True, inplace=True)
df

,file,file_name,total_equations,diff_eq_raw,alg_eq_raw,ode_like_raw,pde_like_raw,state_count,algebraic_count,warnings,...,equivalence_groups,file_hash,file_path,full_path,is_duplicate,duplicate_of,domain,all_domains,n_files_with_same_hash,is_empty_model
0,models_cellml2/calcium_dynamics/dawson_lea_irv...,dawson_lea_irvine_2003.cellml,20,10,10,10,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,"{('environment', 'time'): [('environment', 'ti...",00f0b69613c5024d55c13a07fd0016e69efa7b01a25b97...,calcium_dynamics/dawson_lea_irvine_2003.cellml,models_cellml2/calcium_dynamics/dawson_lea_irv...,False,None,calcium_dynamics,"[calcium_dynamics, signal_transduction]",2,False
1,models_cellml2/circadian_rhythms/goodwin_1965_...,goodwin_1965_a.cellml,2,2,0,2,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,"{('environment', 'time'): [('environment', 'ti...",00f497aaea76b568e3615817f3f0fb35fd5e4000aa4284...,circadian_rhythms/goodwin_1965_a.cellml,models_cellml2/circadian_rhythms/goodwin_1965_...,False,None,circadian_rhythms,[circadian_rhythms],1,False
2,models_cellml2/endocrine/maldonado_2006.cellml,maldonado_2006.cellml,20,9,11,9,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,"{('environment', 'time'): [('environment', 'ti...",03b644bfd5cbd12d5a3910bf8be71b920069b09183c49b...,endocrine/maldonado_2006.cellml,models_cellml2/endocrine/maldonado_2006.cellml,False,None,endocrine,[endocrine],1,False
3,models_cellml2/signal_transduction/riccobene_o...,riccobene_omann_linderman_1999.cellml,8,8,0,8,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,"{('environment', 'time'): [('environment', 'ti...",03e82c8edb463012e5adb801864a1ae837e0885ea49806...,signal_transduction/riccobene_omann_linderman_...,models_cellml2/signal_transduction/riccobene_o...,False,None,signal_transduction,[signal_transduction],1,False
4,models_cellml2/metabolism/olsen_hauser_kummer_...,olsen_hauser_kummer_2003.cellml,29,14,15,14,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,"{('environment', 'time'): [('environment', 'ti...",042bbd67fe92f06a738b898e3257db53c436e4e59fe9c5...,metabolism/olsen_hauser_kummer_2003.cellml,models_cellml2/metabolism/olsen_hauser_kummer_...,False,None,metabolism,[metabolism],1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
482,models_cellml2/electrophysiology/ebihara_johns...,ebihara_johnson_1980.cellml,23,7,16,7,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,"{('environment', 'time'): [('environment', 'ti...",f9f15dab19a2987627df45a9b6a987e364a3612f740f09...,electrophysiology/ebihara_johnson_1980.cellml,models_cellml2/electrophysiology/ebihara_johns...,False,None,electrophysiology,[electrophysiology],1,False
483,models_cellml2/mechanical_constitutive_laws/ri...,rivlin_saunders_1951.cellml,6,0,6,0,0,0,0,libCellML issues present,...,"{('interface', 'E11'): [('interface', 'E11'), ...",fd041499ccc21fa9a3fc59964d21c28dcdb06f71b8341c...,mechanical_constitutive_laws/rivlin_saunders_1...,models_cellml2/mechanical_constitutive_laws/ri...,False,None,mechanical_constitutive_laws,[mechanical_constitutive_laws],1,False
484,models_cellml2/signal_transduction/bagci_2008b...,bagci_2008b.cellml,112,42,70,42,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,"{('environment', 'time'): [('environment', 'ti...",fdf927ecdc71473692ddd5e2dba6b0c9ff98bb0c8a57cc...,signal_transduction/bagci_2008b.cellml,models_cellml2/signal_transduction/bagci_2008b...,False,None,signal_transduction,[signal_transduction],1,False
485,models_cellml2/electrophysiology/maltsev_2009_...,maltsev_2009_paper.cellml,119,29,90,29,0,0,0,Mismatch: state_count vs raw ODE-like equation...,...,"{('environment', 'time'): [('environment', 'ti...",ff33d53fb395b89046506b82ed2d84fa8d658a08da2b55...,electrophysiology/maltsev_2009_paper.cellml,models_cellml2/electrophysiology/maltsev_2009_...,False,None,electrophysiology,[electrophysiology],1,False


In [34]:
table = df[["file_name", "file", "domain", "all_domains", "total_equations", "diff_eq_raw", "alg_eq_raw", "ode_like_raw", "pde_like_raw", "state_count", "algebraic_count", "raw_variable_count", "unique_variable_count"]]

In [35]:
table["file_name"] = table["file_name"].apply(lambda x: x[:-7])

/var/folders/xr/my72_1yx6g53vqntnttc4v640000gn/T/ipykernel_54373/2812429478.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table["file_name"] = table["file_name"].apply(lambda x: x[:-7])


In [36]:
table["file_name"]

0              dawson_lea_irvine_2003
1                      goodwin_1965_a
2                      maldonado_2006
3      riccobene_omann_linderman_1999
4            olsen_hauser_kummer_2003
                    ...              
482              ebihara_johnson_1980
483              rivlin_saunders_1951
484                       bagci_2008b
485                maltsev_2009_paper
486                corrias_buist_2007
Name: file_name, Length: 487, dtype: object

In [37]:
table.tail(4)

,file_name,file,domain,all_domains,total_equations,diff_eq_raw,alg_eq_raw,ode_like_raw,pde_like_raw,state_count,algebraic_count,raw_variable_count,unique_variable_count
483,rivlin_saunders_1951,models_cellml2/mechanical_constitutive_laws/ri...,mechanical_constitutive_laws,[mechanical_constitutive_laws],6,0,6,0,0,0,0,28,14
484,bagci_2008b,models_cellml2/signal_transduction/bagci_2008b...,signal_transduction,[signal_transduction],112,42,70,42,0,0,0,435,203
485,maltsev_2009_paper,models_cellml2/electrophysiology/maltsev_2009_...,electrophysiology,[electrophysiology],119,29,90,29,0,0,0,338,192
486,corrias_buist_2007,models_cellml2/electrophysiology/corrias_buist...,electrophysiology,[electrophysiology],66,14,52,14,0,0,0,227,100


In [38]:
table["ok"] = table["file_name"].duplicated().apply(lambda x: not x)

/var/folders/xr/my72_1yx6g53vqntnttc4v640000gn/T/ipykernel_54373/755218318.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table["ok"] = table["file_name"].duplicated().apply(lambda x: not x)


In [39]:
table = table.loc[table.ok, :]
table.reset_index(inplace=True, drop=True)


In [40]:
table.drop(["ok"], axis=1, inplace=True)

In [41]:
table.columns = ['Model Name', 'File', 'Domain', 'All Domains', 'Equation Count', 'Differential Equation Count', 'Algebraic Equation Count', 'ODE count', 'PDE count', 'State variable count', 'Algebraic Variable Count', 'Total Variable Count', 'Actual Variable Count']

In [42]:
table["All Domains"] = table["All Domains"].apply(
    lambda x: " | ".join(
        sorted(set(d.replace("_", " ") for d in x))
    ) if isinstance(x, (list, tuple)) else ""
)

In [43]:
table

,Model Name,File,Domain,All Domains,Equation Count,Differential Equation Count,Algebraic Equation Count,ODE count,PDE count,State variable count,Algebraic Variable Count,Total Variable Count,Actual Variable Count
0,dawson_lea_irvine_2003,models_cellml2/calcium_dynamics/dawson_lea_irv...,calcium_dynamics,calcium dynamics | signal transduction,20,10,10,10,0,0,0,95,38
1,goodwin_1965_a,models_cellml2/circadian_rhythms/goodwin_1965_...,circadian_rhythms,circadian rhythms,2,2,0,2,0,0,0,13,9
2,maldonado_2006,models_cellml2/endocrine/maldonado_2006.cellml,endocrine,endocrine,20,9,11,9,0,0,0,109,67
3,riccobene_omann_linderman_1999,models_cellml2/signal_transduction/riccobene_o...,signal_transduction,signal transduction,8,8,0,8,0,0,0,80,20
4,olsen_hauser_kummer_2003,models_cellml2/metabolism/olsen_hauser_kummer_...,metabolism,metabolism,29,14,15,14,0,0,0,128,46
...,...,...,...,...,...,...,...,...,...,...,...,...,...
482,ebihara_johnson_1980,models_cellml2/electrophysiology/ebihara_johns...,electrophysiology,electrophysiology,23,7,16,7,0,0,0,59,31
483,rivlin_saunders_1951,models_cellml2/mechanical_constitutive_laws/ri...,mechanical_constitutive_laws,mechanical constitutive laws,6,0,6,0,0,0,0,28,14
484,bagci_2008b,models_cellml2/signal_transduction/bagci_2008b...,signal_transduction,signal transduction,112,42,70,42,0,0,0,435,203
485,maltsev_2009_paper,models_cellml2/electrophysiology/maltsev_2009_...,electrophysiology,electrophysiology,119,29,90,29,0,0,0,338,192


In [44]:
table.to_csv("CellML_model_list_updates.csv", index=False)

## FIX THE PMR FILE DUPLICATES & DOUBLE CHECK THAT REMOVED DUPLICATES ARE ACTUALLY DUPLICATES

In [51]:
table[(table["ODE count"] > 0) & (table["PDE count"] == 0)].reset_index(drop=True).to_csv("CellML_model_list_only_ODE.csv", index=False)